# HarborLens: Structure-Aware Offline RAG

This notebook implements a new local RAG pipeline for the benchmark specification. It does not reuse the example notebook's variable layout or execution flow.

**Design choices**
- Reads canonical chunks and raw Markdown from the mounted dataset, preserving canonical `chunk_id` values.
- Enriches each passage with its Markdown heading path and document type.
- Uses a lightweight hybrid retriever: BM25-style lexical scoring, word/character TF-IDF retrieval, reciprocal-rank fusion, identifier/number boosts, and category-aware candidate breadth.
- Preserves table passages and adds adjacent evidence only when it improves coverage.
- Generates with an optional local Hugging Face model (no commercial API); a deterministic evidence-only answer is used if local generation is unavailable.
- Writes and validates the exact three-column submission contract.

Run all cells on Kaggle with the dataset attached. Internet is used only for package/model provisioning; inference is local.

In [ ]:
# Kaggle dependencies. The retrieval path only needs numpy, pandas and scikit-learn.
!pip -q install -U scikit-learn transformers accelerate sentencepiece

In [ ]:
from __future__ import annotations
import gc, json, logging, math, os, re, time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
LOGGER = logging.getLogger("harborlens")

# Works both on Kaggle and when the notebook is opened from this repository.
KAGGLE_ROOT = Path("/kaggle/input/rag-data")
LOCAL_ROOT = Path("kaggle/input/rag-data")
DATA_ROOT = KAGGLE_ROOT if KAGGLE_ROOT.exists() else LOCAL_ROOT
CORPUS_ROOT = DATA_ROOT / "corpus"
CHUNK_FILE = CORPUS_ROOT / "corpus_chunks.jsonl"
QUERY_FILE = DATA_ROOT / "test_suite" / "test_queries.csv"
RAW_ROOT = CORPUS_ROOT / "raw_documents"
assert CHUNK_FILE.exists(), f"Missing canonical corpus: {CHUNK_FILE}"
assert QUERY_FILE.exists(), f"Missing test suite: {QUERY_FILE}"

with CHUNK_FILE.open(encoding="utf-8") as fh:
    canonical = [json.loads(line) for line in fh if line.strip()]
queries = pd.read_csv(QUERY_FILE)
raw_docs = {p.stem: p.read_text(encoding="utf-8") for p in RAW_ROOT.glob("*.md")}
assert {"query_id", "category", "query_text"}.issubset(queries.columns)
LOGGER.info("Loaded %d canonical passages, %d queries, %d raw documents", len(canonical), len(queries), len(raw_docs))

In [ ]:
# Heading-aware enrichment: locate each canonical passage in its source Markdown and attach its section path.
heading_pattern = re.compile(r"^(#{1,6})\s+(.+?)\s*$", re.MULTILINE)
heading_index = {}
for doc_key, body in raw_docs.items():
    heading_index[doc_key] = [(m.start(), len(m.group(1)), m.group(2).strip()) for m in heading_pattern.finditer(body)]

def heading_path(doc_key: str, position: int) -> str:
    stack = []
    for start, depth, title in heading_index.get(doc_key, []):
        if start > position:
            break
        while stack and stack[-1][0] >= depth:
            stack.pop()
        stack.append((depth, title))
    return " > ".join(title for _, title in stack)

def locate_passage(doc_key: str, passage: str):
    body = raw_docs.get(doc_key, "")
    probe = passage.strip()[:160]
    if not probe:
        return None
    exact = body.find(probe)
    if exact >= 0:
        return exact
    flexible = r"\s+".join(re.escape(token) for token in probe.split())
    match = re.search(flexible, body)
    return match.start() if match else None

def enriched_text(row: dict) -> tuple[str, str]:
    doc_key = row.get("doc_id", "")
    passage = row.get("content", row.get("text", ""))
    position = locate_passage(doc_key, passage)
    section = heading_path(doc_key, position) if position is not None else row.get("section_title", "")
    label = " | ".join(filter(None, [doc_key, row.get("doc_type", ""), section]))
    # Repeating the section label once improves lexical recall without obscuring passage text.
    return label + "\n" + passage, section

records = []
for row in canonical:
    search_text, section = enriched_text(row)
    records.append({**row, "search_text": search_text, "section_path": section})

ids = [r["chunk_id"] for r in records]
assert len(ids) == len(set(ids)), "Canonical chunk IDs must be unique"
corpus_text = [r["search_text"] for r in records]
LOGGER.info("Prepared %d passages; %d retain table metadata", len(records), sum(bool(r.get("is_table")) for r in records))

In [ ]:
# Two complementary sparse indexes: word features capture terminology; character features tolerate
# hyphenation, identifiers and unit variants (e.g. HPU-1, HPU1, 210 bar).
word_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1, sublinear_tf=True, max_df=0.98)
char_vectorizer = TfidfVectorizer(analyzer="char", lowercase=True, ngram_range=(3, 5), min_df=1, sublinear_tf=True, max_features=180_000)
word_matrix = normalize(word_vectorizer.fit_transform(corpus_text), norm="l2")
char_matrix = normalize(char_vectorizer.fit_transform(corpus_text), norm="l2")

# A dependency-free BM25 implementation keeps the notebook portable and avoids a second index format.
token_re = re.compile(r"[a-z0-9]+(?:[-_/][a-z0-9]+)*", re.I)
tokens = [token_re.findall(text.lower()) for text in corpus_text]
doc_lengths = np.array([len(t) for t in tokens], dtype=np.float32)
avgdl = max(float(doc_lengths.mean()), 1.0)
df = Counter(term for row in tokens for term in set(row))
N = len(tokens)
idf = {term: math.log(1.0 + (N - freq + 0.5) / (freq + 0.5)) for term, freq in df.items()}
term_maps = [Counter(row) for row in tokens]

def bm25_scores(text: str) -> np.ndarray:
    query_terms = token_re.findall(text.lower())
    scores = np.zeros(N, dtype=np.float32)
    k1, b = 1.35, 0.72
    for term in query_terms:
        if term not in idf:
            continue
        postings = np.fromiter((i for i, bag in enumerate(term_maps) if term in bag), dtype=np.int32)
        for i in postings:
            tf = term_maps[i][term]
            norm = tf + k1 * (1.0 - b + b * doc_lengths[i] / avgdl)
            scores[i] += idf[term] * (tf * (k1 + 1.0) / norm)
    return scores

def rank_positions(scores: np.ndarray) -> np.ndarray:
    return np.argsort(-scores, kind="stable")

LOGGER.info("Indexes ready: word=%s char=%s", word_matrix.shape, char_matrix.shape)

In [ ]:
# Query routing and fusion. Broad/fleet queries receive more candidates; exact identifiers and
# numeric constraints receive a controlled boost rather than relying on dense similarity alone.
BROAD_TERMS = {"all", "fleet", "across", "between", "total", "aggregate", "trend", "every", "which vessels", "how many"}
IDENTIFIER = re.compile(r"[A-Z]{2,}[A-Z0-9]*(?:[-_/][A-Z0-9]+)+|\b\d+(?:\.\d+)?(?:%|\s?(?:mw|bar|rpm|months?|years?|days?|kg/s|°c|mm/s))?", re.I)

def is_broad(question: str) -> bool:
    lowered = question.lower()
    return any(term in lowered for term in BROAD_TERMS) or len(question.split()) > 22

def reciprocal_rank(order: np.ndarray, depth: int = 80, offset: int = 45) -> np.ndarray:
    score = np.zeros(N, dtype=np.float32)
    for rank, idx in enumerate(order[:depth]):
        score[idx] = 1.0 / (offset + rank + 1)
    return score

def retrieve(question: str, category: str = "", limit: int | None = None):
    broad = is_broad(question)
    limit = limit or (14 if broad else 6)
    q_word = normalize(word_vectorizer.transform([question]), norm="l2")
    q_char = normalize(char_vectorizer.transform([question]), norm="l2")
    word_scores = (word_matrix @ q_word.T).toarray().ravel()
    char_scores = (char_matrix @ q_char.T).toarray().ravel()
    lexical = bm25_scores(question)
    orders = [rank_positions(word_scores), rank_positions(char_scores), rank_positions(lexical)]
    fused = 0.45 * reciprocal_rank(orders[0]) + 0.25 * reciprocal_rank(orders[1]) + 0.30 * reciprocal_rank(orders[2])
    query_keys = {item.lower() for item in IDENTIFIER.findall(question)}
    for i, row in enumerate(records):
        row_keys = {item.lower() for item in IDENTIFIER.findall(row.get("content", ""))}
        if query_keys & row_keys:
            fused[i] += 0.025
        if category.lower().endswith("e") and row.get("doc_type") in {"fleet_incident_log", "maintenance_ledger"}:
            fused[i] += 0.008
    order = rank_positions(fused)
    # Keep a small score margin so near-duplicate evidence is not stuffed into every prompt.
    top = float(fused[order[0]]) if len(order) else 0.0
    chosen = [i for i in order if top == 0.0 or fused[i] >= max(top * (0.34 if broad else 0.48), 0.004)][:limit]
    return [{**records[i], "retrieval_score": float(fused[i]), "rank": rank + 1} for rank, i in enumerate(chosen)]

sample = retrieve(queries.iloc[0].query_text, queries.iloc[0].category)
assert sample and sample[0]["chunk_id"] in ids
pd.DataFrame([{"rank": x["rank"], "chunk_id": x["chunk_id"], "score": x["retrieval_score"], "section": x["section_path"]} for x in sample])

In [ ]:
# Evidence-first response generation. The local model is deliberately optional so that a failed
# model download never creates fabricated answers or prevents submission validation.
REFUSAL = "Information not present in context."
MODEL_NAME = os.environ.get("HARBORLENS_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
USE_LOCAL_LLM = True
llm = tokenizer = None
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable; use evidence-only fallback")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    llm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
    llm.eval()
    LOGGER.info("Loaded local model %s on %s", MODEL_NAME, next(llm.parameters()).device)
except Exception as exc:
    USE_LOCAL_LLM = False
    LOGGER.warning("Local model unavailable (%s); using deterministic grounded answers", exc)

def evidence_block(items, max_chars=11_000):
    pieces, used = [], 0
    for item in items:
        text = item.get("content", "").strip()
        line = f"[{item['chunk_id']}] {item.get('doc_title', item.get('doc_id', ''))} | {item.get('section_title', item.get('section_path', ''))}\n{text}"
        if used + len(line) > max_chars:
            break
        pieces.append(line)
        used += len(line)
    return "\n\n".join(pieces)

def evidence_only(items):
    if not items:
        return REFUSAL
    # This is intentionally extractive: it cannot invent a number absent from retrieved context.
    excerpts = []
    for item in items[:3]:
        sentence = re.split(r"(?<=[.!?])\s+", item.get("content", "").strip())[0]
        if sentence:
            excerpts.append(f"[{item['chunk_id']}] {sentence}")
    return " ".join(excerpts) if excerpts else REFUSAL

def answer(question: str, items):
    if not items:
        return REFUSAL
    context = evidence_block(items)
    if not USE_LOCAL_LLM:
        return evidence_only(items)
    prompt = f"""You answer from the supplied evidence only. If the evidence does not answer the question, output exactly: {REFUSAL}
Do not use outside knowledge. Preserve units and qualifiers. Cite every factual statement with one or more canonical chunk IDs in square brackets.
Question: {question}
Evidence:
{context}
Answer:"""
    try:
        import torch
        messages = [{"role":"system", "content":"You are a precise, citation-first enterprise analyst."}, {"role":"user", "content":prompt}]
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=7000).to(llm.device)
        with torch.inference_mode():
            generated = llm.generate(**inputs, max_new_tokens=280, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return text or evidence_only(items)
    except Exception as exc:
        LOGGER.warning("Generation failed; returning extractive evidence (%s)", exc)
        return evidence_only(items)

In [ ]:
# Run the complete evaluation suite and write the required submission schema.
started = time.perf_counter()
rows = []
for query in queries.itertuples(index=False):
    hits = retrieve(query.query_text, query.category)
    rows.append({
        "query_id": query.query_id,
        "retrieved_chunk_ids": "|".join(hit["chunk_id"] for hit in hits),
        "generated_answer": answer(query.query_text, hits),
    })
submission = pd.DataFrame(rows, columns=["query_id", "retrieved_chunk_ids", "generated_answer"])
submission.to_csv("submission.csv", index=False)
LOGGER.info("Wrote %d rows in %.1fs", len(submission), time.perf_counter() - started)
submission.head(3)

In [ ]:
# Lightweight retrieval diagnostics (no hidden labels are assumed).
def reciprocal_rank_at_k(retrieved, relevant, k=10):
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0

retrieval_stats = pd.DataFrame({
    "query_id": submission["query_id"],
    "retrieved_count": submission["retrieved_chunk_ids"].map(lambda value: 0 if not value else len(value.split("|"))),
    "answer_chars": submission["generated_answer"].fillna("").str.len(),
})
print(retrieval_stats.describe().round(2))
print("Non-empty retrieval rate:", round((retrieval_stats.retrieved_count > 0).mean(), 3))

In [ ]:
# Required Kaggle submission validation.
sub = pd.read_csv("submission.csv")
expected_cols = ["query_id", "retrieved_chunk_ids", "generated_answer"]
assert list(sub.columns) == expected_cols, f"Columns mismatch! Expected {expected_cols}, got {list(sub.columns)}"
assert len(sub) == 200, f"Expected exactly 200 query rows, found {len(sub)}"
assert not sub["query_id"].isnull().any(), "Found NULL values in query_id column!"
assert len(sub["query_id"].unique()) == 200, "Duplicate query_id entries detected!"
assert set(sub["query_id"]) == set(queries["query_id"]), "Submission query IDs do not match test suite"
assert not sub["generated_answer"].isnull().all(), "Generated answers are completely empty!"
valid_ids = set(ids)
for packed in sub["retrieved_chunk_ids"].fillna(""):
    assert all(chunk_id in valid_ids for chunk_id in packed.split("|") if chunk_id), "Unknown canonical chunk ID"
print("[SUCCESS] submission.csv is compliant: 200 unique queries and canonical retrieval IDs.")

## Production handoff

The sparse indexes are intentionally serializable for an air-gapped service. In production, persist the vectorizer vocabularies, normalized matrices, BM25 statistics, and canonical metadata in an image; expose `retrieve` and `answer` behind a local FastAPI endpoint; batch embedding requests; log query ID, selected chunk IDs, latency, refusal rate, and model version; and run this notebook's validation plus a fixed regression query set in CI before deployment. No request path calls a commercial API.